# Kaggle runner: training (C3) and generation

Same scripts as the Colab notebooks; only paths and setup differ. One-time setup on Kaggle:
1. Datasets > New Dataset: upload `spider_bundle.zip` (private). Kaggle may unpack it; both cases work.
2. In this notebook: Add Input > your dataset. Settings: Accelerator **GPU T4 x2**, Internet **on**
   (needs a verified phone number).
3. Edit the queue cell, then **Save Version > Save & Run All** to run in the background (up to 12 h).
   Outputs appear in the version's Output tab; download `outputs/` from there.

Only GPU 0 is used, so the effective batch stays 16 exactly as on Colab (SPEC 6).

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4, same effective batch as Colab
!rm -rf /tmp/repo && git clone -q https://github.com/frangarciaproff/spider-sql-lora-ladder.git /tmp/repo
%cd /tmp/repo
!git log -1 --format="%H %s"
!pip install -q -e . peft
!pip install -q unsloth
!python --version

In [ ]:
import glob

BUNDLE_SHA256 = "aacfdd35b1a6afc369ef028cff59c39ac8639fd813c3b092fb7e5a1615e2b44f"
zips = glob.glob("/kaggle/input/**/spider_bundle.zip", recursive=True)
manifests = glob.glob("/kaggle/input/**/MANIFEST.json", recursive=True)
if zips:
    !python scripts/get_data.py --zip {zips[0]} --dest /tmp/data --sha256 {BUNDLE_SHA256}
    DATA = "/tmp/data"
elif manifests:
    DATA = os.path.dirname(manifests[0])
    !python scripts/get_data.py --extracted {DATA}
else:
    raise FileNotFoundError("add the spider bundle dataset as an input")
os.environ["SPIDER_DIR"] = f"{DATA}/spider"
os.environ["TESTSUITE_DIR"] = f"{DATA}/testsuite"

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2"
print(torch.cuda.get_device_name(0), torch.cuda.device_count(), "visible")

MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
TAG = "qwen05b"
PER_DEVICE, ACCUM = 8, 2  # effective batch 16; use 4, 4 for 1.5B
GEN_BATCH = 32  # 16 for 1.5B
BACKEND = "unsloth"  # same backend as the Colab grid runs
OUT = "/kaggle/working/outputs"
ADAPTERS, PRED, RES = f"{OUT}/adapters", f"{OUT}/predictions", f"{OUT}/results"

In [ ]:
def check(step):
    if _exit_code != 0:  # noqa: F821  (set by IPython after each ! command)
        raise RuntimeError(f"{step} failed; see the output above")


def train_and_validate(lr, epochs, seed):
    run = f"{TAG}_c3_lr{lr:g}_ep{epochs}_seed{seed}"
    !python scripts/train.py --model {MODEL} --backend {BACKEND} --per-device-batch {PER_DEVICE} --grad-accum {ACCUM} --lr {lr} --epochs {epochs} --seed {seed} --out-dir {ADAPTERS}/{run}
    check(f"training {run}")
    !rm -rf {ADAPTERS}/{run}/checkpoint-*
    generate_and_score(run, "adapter", "validation", adapter=f"{ADAPTERS}/{run}/adapter")
    return run


def generate_and_score(run, condition, split, adapter=None, final=False):
    extra = (f"--adapter {adapter}" if adapter else "") + (" --final" if final else "")
    !python scripts/generate.py --model {MODEL} --condition {condition} --split {split} --batch-size {GEN_BATCH} --out-dir {PRED}/{run} {extra}
    check(f"generation {run} {split}")
    !python scripts/score_predictions.py --pred-dir {PRED}/{run} --split {split} --out-dir {RES}/{run}
    check(f"scoring {run} {split}")

## Queue for this version
Edit only this cell between versions. Now: the last grid point (0.5B, seed 0).

In [ ]:
train_and_validate(2e-4, 3, seed=0)

In [ ]:
!find {OUT} -maxdepth 3 -name "*.json" | sort